# Notebook C1 — TinyPCGNet v4.1 Training + Final Comparison

**Purpose:** train TinyPCGNet v4.1 on the *same* 5 folds as the 21 baselines
(from Notebook A's `oahs_folds.npz`), then merge results into a single final
comparison table with all 22 rows.

This notebook produces three deliverables:

1. **`tinypcgnet_v41_best.keras`** — best fold's saved model (input to Notebook C2)
2. **`results_master_final.csv`** — 22-row comparison table (21 baselines + v4.1)
3. **`final_comparison_plots/`** — headline plots for your report

After this runs, Notebook C2 picks up the saved model for Grad-CAM analysis.

## Apples-to-apples guarantee
- Same fold indices as Notebook B (`oahs_folds.npz`)
- Same LR schedule, optimizer, batch size, EarlyStopping
- Same metrics computed the same way
- Same FLOPs estimator


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
import os, time, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, LearningRateScheduler
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

## 2. Configuration

Training settings deliberately match Notebook B's so the v4.1 row in the
final table is fully comparable to the 21 baseline rows.

In [ ]:
# ---------- Paths ----------
FEAT_DIR   = '/content/drive/MyDrive/Msc_ML_project/features_v1'
RESULTS_B  = '/content/drive/MyDrive/Msc_ML_project/results_v1'  # Notebook B's outputs
OUTPUT_DIR = '/content/drive/MyDrive/Msc_ML_project/results_v1_final'  # C1's outputs
os.makedirs(OUTPUT_DIR, exist_ok=True)
PLOT_DIR = os.path.join(OUTPUT_DIR, 'final_comparison_plots')
os.makedirs(PLOT_DIR, exist_ok=True)
print('OUTPUT_DIR:', OUTPUT_DIR)

# ---------- Classes ----------
CLASSES = ['AS', 'MR', 'MS', 'MVP', 'N']
NUM_CLASSES = len(CLASSES)

# ---------- TinyPCGNet architecture ----------
MFCC_WIDTHS  = [16, 24]
MFCC_POOLS   = 2
SPEC_WIDTHS  = [16, 24, 32]
SPEC_POOLS   = 2
KERNEL       = (3, 3)
SE_RATIO     = 4
FUSION_UNITS = 32
DROPOUT      = 0.3

# ---------- Training (matches Notebook B) ----------
EPOCHS          = 100
BATCH_SIZE      = 32
LR_BASE         = 3e-4
LR_WARMUP_INIT  = 5e-5
WARMUP_EPOCHS   = 5
ES_PATIENCE     = 20
ES_MIN_EPOCHS   = 25
RLR_PATIENCE    = 10

# Retry on collapse (only for v4.1; baselines didn't use this)
COLLAPSE_THRESHOLD = 0.50
MAX_RETRIES        = 2

EPS = 1e-10
print('Config loaded.')

## 3. Load cached features and shared folds

In [ ]:
mfcc_d = np.load(os.path.join(FEAT_DIR, 'oahs_mfcc.npz'))
spec_d = np.load(os.path.join(FEAT_DIR, 'oahs_logmel.npz'))
X_mfcc = mfcc_d['X']
X_spec = spec_d['X']
y      = mfcc_d['y']
files  = mfcc_d['files']

# Sanity: both feature arrays must align on samples and labels
assert np.array_equal(mfcc_d['y'], spec_d['y']), 'MFCC and LogMel labels disagree!'
assert X_mfcc.shape[0] == X_spec.shape[0]

folds = np.load(os.path.join(FEAT_DIR, 'oahs_folds.npz'), allow_pickle=True)
fold_train = [np.asarray(a, dtype=np.int64) for a in folds['train_idx']]
fold_val   = [np.asarray(a, dtype=np.int64) for a in folds['val_idx']]

MFCC_INPUT_SHAPE = X_mfcc.shape[1:]
SPEC_INPUT_SHAPE = X_spec.shape[1:]

print(f'MFCC: {X_mfcc.shape}  shape per sample={MFCC_INPUT_SHAPE}')
print(f'Spec: {X_spec.shape}  shape per sample={SPEC_INPUT_SHAPE}')
print(f'y   : {y.shape}  classes: {np.bincount(y).tolist()}')
print(f'Folds: {len(fold_train)}  (val sizes: {[len(v) for v in fold_val]})')

## 4. Build TinyPCGNet v4.1

In [ ]:
def squeeze_excite_block(x, ratio=SE_RATIO, name=''):
    filters = x.shape[-1]
    se = layers.GlobalAveragePooling2D(name=f'{name}_gap')(x)
    se = layers.Dense(max(filters // ratio, 4), activation='relu',
                      use_bias=False, name=f'{name}_squeeze')(se)
    se = layers.Dense(filters, activation='sigmoid',
                      use_bias=False, name=f'{name}_excite')(se)
    se = layers.Reshape((1, 1, filters), name=f'{name}_reshape')(se)
    return layers.Multiply(name=f'{name}_scale')([x, se])


def ds_conv_block(x, filters, pool=True, prefix='block'):
    x = layers.SeparableConv2D(filters, KERNEL, padding='same',
                               use_bias=False, name=f'{prefix}_sepconv')(x)
    x = layers.BatchNormalization(name=f'{prefix}_bn')(x)
    x = layers.ReLU(max_value=6.0, name=f'{prefix}_relu6')(x)
    if pool:
        x = layers.MaxPooling2D((2, 2), padding='same', name=f'{prefix}_pool')(x)
    return x


def build_mfcc_branch(inp):
    x = inp
    for i, w in enumerate(MFCC_WIDTHS):
        pool = (i < MFCC_POOLS)
        x = ds_conv_block(x, w, pool=pool, prefix=f'mfcc_b{i+1}')
    x = squeeze_excite_block(x, name='mfcc_se')
    x = layers.GlobalAveragePooling2D(name='mfcc_gap')(x)
    return x


def build_spec_branch(inp):
    x = inp
    for i, w in enumerate(SPEC_WIDTHS):
        pool = (i < SPEC_POOLS)
        x = ds_conv_block(x, w, pool=pool, prefix=f'spec_b{i+1}')
    x = squeeze_excite_block(x, name='spec_se')
    x = layers.GlobalAveragePooling2D(name='spec_gap')(x)
    return x


def build_tinypcgnet_v41(mfcc_shape=MFCC_INPUT_SHAPE,
                         spec_shape=SPEC_INPUT_SHAPE,
                         num_classes=NUM_CLASSES):
    mfcc_in = Input(shape=mfcc_shape, name='mfcc_input')
    spec_in = Input(shape=spec_shape, name='spec_input')
    mfcc_feat = build_mfcc_branch(mfcc_in)
    spec_feat = build_spec_branch(spec_in)
    fused = layers.Concatenate(name='fusion_concat')([mfcc_feat, spec_feat])
    fused = layers.Dense(FUSION_UNITS, activation='relu', name='fusion_dense')(fused)
    fused = layers.Dropout(DROPOUT, name='fusion_dropout')(fused)
    out   = layers.Dense(num_classes, activation='softmax', name='probs')(fused)
    return Model([mfcc_in, spec_in], out, name='TinyPCGNet_v41')


probe = build_tinypcgnet_v41()
probe.summary()
n_params_v41 = int(sum(np.prod(v.shape) for v in probe.trainable_weights))
print(f'\n>>> Trainable params: {n_params_v41:,}')

## 5. FLOPs estimator (Keras-3 clean — matches Notebook B)

In [ ]:
def _layer_input_shape(layer):
    try:
        s = layer.input.shape
    except (AttributeError, TypeError):
        s = getattr(layer, 'input_shape', None)
    if s is None: return None
    if isinstance(s, list): s = s[0]
    return tuple(s)


def _layer_output_shape(layer):
    try:
        s = layer.output.shape
    except (AttributeError, TypeError):
        s = getattr(layer, 'output_shape', None)
    if s is None: return None
    if isinstance(s, list): s = s[0]
    return tuple(s)


def estimate_flops(model):
    total = 0
    for layer in model.layers:
        if isinstance(layer, (layers.InputLayer, layers.Reshape,
                              layers.Permute, layers.Flatten,
                              layers.Dropout, layers.Activation,
                              layers.ReLU, layers.GlobalAveragePooling2D,
                              layers.MaxPooling2D, layers.AveragePooling2D,
                              layers.Multiply, layers.Concatenate)):
            continue
        in_shape  = _layer_input_shape(layer)
        out_shape = _layer_output_shape(layer)
        if in_shape is None or out_shape is None: continue

        if isinstance(layer, layers.SeparableConv2D):
            H_out, W_out, C_out = out_shape[1], out_shape[2], out_shape[3]
            kH, kW = layer.kernel_size
            C_in   = in_shape[3]
            total += 2 * H_out * W_out * C_in * kH * kW
            total += 2 * H_out * W_out * C_out * C_in
        elif isinstance(layer, layers.Conv2D):
            H_out, W_out, C_out = out_shape[1], out_shape[2], out_shape[3]
            kH, kW = layer.kernel_size
            C_in   = in_shape[3]
            total += 2 * H_out * W_out * C_out * kH * kW * C_in
        elif isinstance(layer, layers.Dense):
            n_in, n_out = in_shape[-1], out_shape[-1]
            total += 2 * n_in * n_out
        elif isinstance(layer, layers.BatchNormalization):
            shape = out_shape[1:]
            elems = 1
            for s in shape:
                if s is not None: elems *= s
            total += 2 * elems
    return int(total)


flops_v41 = estimate_flops(probe)
print(f'TinyPCGNet v4.1: ~{flops_v41/1e6:.3f} MFLOPs, {n_params_v41:,} params')
del probe; tf.keras.backend.clear_session()

## 6. Training utilities

Reusing the same warmup + protected EarlyStopping + retry-on-collapse
pattern from v4.1.

In [ ]:
def warmup_schedule(epoch, lr):
    if epoch < WARMUP_EPOCHS:
        frac = (epoch + 1) / WARMUP_EPOCHS
        return float(LR_WARMUP_INIT + (LR_BASE - LR_WARMUP_INIT) * frac)
    return float(lr)


class MinEpochsEarlyStopping(EarlyStopping):
    def __init__(self, min_epochs=0, **kwargs):
        super().__init__(**kwargs)
        self.min_epochs = min_epochs

    def on_epoch_end(self, epoch, logs=None):
        if epoch < self.min_epochs:
            self.wait = 0
            return
        super().on_epoch_end(epoch, logs)


def train_one_attempt(Xm_tr, Xs_tr, y_tr,
                      Xm_val, Xs_val, y_val,
                      fold_idx, attempt):
    tf.keras.backend.clear_session()
    seed = SEED + fold_idx * 100 + attempt
    tf.random.set_seed(seed); np.random.seed(seed); random.seed(seed)

    model = build_tinypcgnet_v41()
    model.compile(
        optimizer=Adam(learning_rate=LR_WARMUP_INIT),
        loss='categorical_crossentropy',
        metrics=['accuracy'])

    y_tr_oh  = to_categorical(y_tr,  NUM_CLASSES).astype(np.float32)
    y_val_oh = to_categorical(y_val, NUM_CLASSES).astype(np.float32)

    callbacks = [
        LearningRateScheduler(warmup_schedule, verbose=0),
        MinEpochsEarlyStopping(
            min_epochs=ES_MIN_EPOCHS,
            monitor='val_accuracy', patience=ES_PATIENCE,
            restore_best_weights=True, mode='max'),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=RLR_PATIENCE, min_lr=1e-6),
    ]

    t0 = time.time()
    hist = model.fit(
        [Xm_tr, Xs_tr], y_tr_oh,
        validation_data=([Xm_val, Xs_val], y_val_oh),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        callbacks=callbacks, verbose=0)
    train_time = time.time() - t0

    y_pred = model.predict([Xm_val, Xs_val], batch_size=BATCH_SIZE,
                           verbose=0).argmax(axis=1)
    metrics = {
        'fold': fold_idx,
        'attempt': attempt,
        'seed': seed,
        'acc':  accuracy_score(y_val, y_pred),
        'prec': precision_score(y_val, y_pred, average='macro', zero_division=0),
        'rec':  recall_score(y_val, y_pred, average='macro', zero_division=0),
        'f1':   f1_score(y_val, y_pred, average='macro', zero_division=0),
        'epochs_trained': len(hist.history['loss']),
        'train_time_s': train_time,
    }
    cm = confusion_matrix(y_val, y_pred, labels=list(range(NUM_CLASSES)))
    return model, metrics, cm, hist.history


def train_fold_with_retry(Xm_tr, Xs_tr, y_tr,
                          Xm_val, Xs_val, y_val, fold_idx):
    best = None
    for attempt in range(MAX_RETRIES + 1):
        print(f'    attempt {attempt+1}/{MAX_RETRIES+1} ...', end=' ', flush=True)
        m, met, cm, hist = train_one_attempt(
            Xm_tr, Xs_tr, y_tr, Xm_val, Xs_val, y_val, fold_idx, attempt)
        print(f"acc={met['acc']:.4f}  ep={met['epochs_trained']}")
        if best is None or met['acc'] > best[1]['acc']:
            best = (m, met, cm, hist)
        if met['acc'] >= COLLAPSE_THRESHOLD:
            return best
        print(f'    -> below threshold {COLLAPSE_THRESHOLD}, retrying')
    return best

print('Training utilities defined.')

## 7. 5-fold cross-validation

Uses the **same fold splits** as Notebook B's 21 baselines.

In [ ]:
all_metrics, all_cms, all_histories = [], [], []
best_model, best_acc, best_fold_idx, best_val_idx = None, -1.0, None, None

t_grid_start = time.time()
for fold_i, (tr_idx, va_idx) in enumerate(zip(fold_train, fold_val), start=1):
    print(f'\n=== Fold {fold_i}/{len(fold_train)} ===')
    Xm_tr, Xm_val = X_mfcc[tr_idx], X_mfcc[va_idx]
    Xs_tr, Xs_val = X_spec[tr_idx], X_spec[va_idx]
    y_tr,  y_val  = y[tr_idx], y[va_idx]

    model, met, cm, hist = train_fold_with_retry(
        Xm_tr, Xs_tr, y_tr, Xm_val, Xs_val, y_val, fold_i)

    print(f"  FINAL: acc={met['acc']:.4f}  f1={met['f1']:.4f}  "
          f"attempts={met['attempt']+1}  time={met['train_time_s']:.1f}s")
    all_metrics.append(met)
    all_cms.append(cm)
    all_histories.append(hist)

    if met['acc'] > best_acc:
        best_acc = met['acc']
        best_model = model
        best_fold_idx = fold_i
        best_val_idx = va_idx

grid_time = time.time() - t_grid_start
print(f'\nAll folds done in {grid_time/60:.1f} min')
print(f'Best fold: {best_fold_idx} with acc={best_acc:.4f}')

## 8. Save best model + per-fold results

The saved best model is the input to Notebook C2 (Grad-CAM analysis).

In [ ]:
# Save best model — this is what Notebook C2 will load
best_model_path = os.path.join(OUTPUT_DIR, 'tinypcgnet_v41_best.keras')
best_model.save(best_model_path)
print(f'Saved best model: {best_model_path} '
      f'({os.path.getsize(best_model_path)/1024:.1f} KB)')

# Per-fold metrics
fold_df = pd.DataFrame(all_metrics)
fold_df.to_csv(os.path.join(OUTPUT_DIR, 'v41_per_fold.csv'), index=False)
print('\nPer-fold metrics:')
print(fold_df.to_string(index=False))

# Aggregate CM across folds
cm_total_v41 = np.sum(all_cms, axis=0)
np.savez_compressed(
    os.path.join(OUTPUT_DIR, 'v41_artifacts.npz'),
    cm_total=cm_total_v41,
    best_fold=best_fold_idx,
    best_val_idx=best_val_idx,
    all_cms=np.array(all_cms),
)

# Histories
with open(os.path.join(OUTPUT_DIR, 'v41_histories.json'), 'w') as f:
    json.dump(all_histories, f)

print('\nSummary:')
for col in ['acc', 'prec', 'rec', 'f1']:
    arr = fold_df[col].values
    print(f'  {col:>5}: {arr.mean():.4f} ± {arr.std():.4f}')

print('\nAggregate confusion matrix:')
print(pd.DataFrame(cm_total_v41, index=CLASSES, columns=CLASSES))

## 9. Merge into final 22-row comparison table

In [ ]:
# Load Notebook B's 21-row table
b_csv = os.path.join(RESULTS_B, 'results_master.csv')
results_b = pd.read_csv(b_csv)
print(f'Loaded {len(results_b)} baseline rows from {b_csv}')
assert len(results_b) == 21, \
    f'Expected 21 baseline rows, got {len(results_b)}. Did Notebook B finish?'

# Build the v4.1 row in the same schema
acc_arr  = fold_df['acc'].values
prec_arr = fold_df['prec'].values
rec_arr  = fold_df['rec'].values
f1_arr   = fold_df['f1'].values
time_arr = fold_df['train_time_s'].values
ep_arr   = fold_df['epochs_trained'].values

v41_row = {
    'model':       'TinyPCGNet-v4.1',
    'feature':     'MFCC+LogMel',
    'params':      n_params_v41,
    'flops':       flops_v41,
    'acc_mean':    float(acc_arr.mean()),
    'acc_std':     float(acc_arr.std()),
    'prec_mean':   float(prec_arr.mean()),
    'prec_std':    float(prec_arr.std()),
    'rec_mean':    float(rec_arr.mean()),
    'rec_std':     float(rec_arr.std()),
    'f1_mean':     float(f1_arr.mean()),
    'f1_std':      float(f1_arr.std()),
    'mean_time_s': float(time_arr.mean()),
    'mean_epochs': float(ep_arr.mean()),
}

# Concat into final table
final_df = pd.concat([results_b, pd.DataFrame([v41_row])], ignore_index=True)
final_csv = os.path.join(OUTPUT_DIR, 'results_master_final.csv')
final_df.to_csv(final_csv, index=False)
print(f'\nFinal table saved: {final_csv}')
print(f'Total rows: {len(final_df)}')

# Display the final table
print('\nFinal 22-row comparison:')
display_cols = ['model', 'feature', 'params', 'flops',
                'acc_mean', 'acc_std', 'f1_mean']
print(final_df[display_cols].to_string(index=False))

## 10. Final comparison plots

Three deliverable plots for your report:
- **Heatmap** (7 baseline models × 3 features, with v4.1 noted separately)
- **Params vs Accuracy** scatter
- **FLOPs vs Accuracy** scatter

v4.1 is highlighted with a special marker to make its position visible.

In [ ]:
# Split for plotting
baselines = final_df[final_df['model'] != 'TinyPCGNet-v4.1'].copy()
v41_only  = final_df[final_df['model'] == 'TinyPCGNet-v4.1'].copy()

GRID_MODELS = ['CNN-1', 'CNN-2', 'SepCNN-1', 'SepCNN-2',
               'Conv-LSTM', 'Conv-BiLSTM', 'Conv-GRU']
GRID_FEATURES = ['DWT', 'MFCC', 'LogMel']

# --------- Heatmap of baselines ---------
pivot_acc = baselines.pivot(index='model', columns='feature',
                            values='acc_mean').reindex(GRID_MODELS)[GRID_FEATURES]
pivot_f1  = baselines.pivot(index='model', columns='feature',
                            values='f1_mean').reindex(GRID_MODELS)[GRID_FEATURES]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, mat, title in [(axes[0], pivot_acc, 'Accuracy'),
                       (axes[1], pivot_f1,  'Macro F1')]:
    vmin = min(mat.values.min(), 0.5)
    im = ax.imshow(mat.values, cmap='viridis', vmin=vmin, vmax=1.0, aspect='auto')
    ax.set_xticks(range(len(GRID_FEATURES))); ax.set_xticklabels(GRID_FEATURES)
    ax.set_yticks(range(len(GRID_MODELS))); ax.set_yticklabels(GRID_MODELS)
    ax.set_title(title)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat.values[i, j]
            ax.text(j, i, f'{v:.3f}', ha='center', va='center',
                    color='white' if v < (vmin + 1.0) / 2 else 'black',
                    fontsize=9)
    plt.colorbar(im, ax=ax)
plt.suptitle(f'Baselines  |  TinyPCGNet-v4.1 = {v41_only.iloc[0]["acc_mean"]:.3f} acc',
             y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'heatmap_baselines_with_v41_in_title.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --------- Params vs Accuracy + FLOPs vs Accuracy ---------
from matplotlib.lines import Line2D

markers = {'DWT': 'o', 'MFCC': 's', 'LogMel': '^', 'MFCC+LogMel': '*'}
colors  = plt.cm.tab10(np.linspace(0, 1, len(GRID_MODELS) + 1))
model_color = dict(zip(GRID_MODELS + ['TinyPCGNet-v4.1'], colors))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, x_col, x_label in [(axes[0], 'params', 'Trainable parameters'),
                           (axes[1], 'flops',  'FLOPs')]:
    for _, row in final_df.iterrows():
        is_v41 = row['model'] == 'TinyPCGNet-v4.1'
        ax.scatter(row[x_col], row['acc_mean'],
                   marker=markers[row['feature']],
                   color=model_color[row['model']],
                   s=180 if is_v41 else 80,
                   edgecolor='red' if is_v41 else 'black',
                   linewidth=2 if is_v41 else 0.5,
                   zorder=5 if is_v41 else 3)
        if is_v41:
            ax.annotate('TinyPCGNet-v4.1',
                       xy=(row[x_col], row['acc_mean']),
                       xytext=(8, -8), textcoords='offset points',
                       fontsize=10, fontweight='bold', color='red')
    ax.set_xscale('log')
    ax.set_xlabel(x_label); ax.set_ylabel('Mean accuracy')
    ax.set_title(f'Accuracy vs {x_label}')
    ax.grid(alpha=0.3)

# Legend — models (colors)
model_handles = [Line2D([0], [0], marker='o', color='w',
                        markerfacecolor=model_color[m], markersize=8, label=m)
                 for m in GRID_MODELS + ['TinyPCGNet-v4.1']]
feat_handles = [Line2D([0], [0], marker=markers[f], color='gray',
                       markersize=8, linestyle='', label=f)
                for f in GRID_FEATURES + ['MFCC+LogMel']]
axes[0].legend(handles=model_handles, loc='lower right',
               fontsize=8, ncol=2, framealpha=0.9)
axes[1].legend(handles=feat_handles, loc='lower right',
               fontsize=8, framealpha=0.9)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'scatter_efficiency_with_v41.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --------- Aggregate confusion matrix for v4.1 ---------
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm_total_v41, cmap='Blues')
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASSES)
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASSES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'TinyPCGNet v4.1 — Aggregate Confusion Matrix\n'
             f'(acc = {acc_arr.mean():.4f} ± {acc_arr.std():.4f})')
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        v = int(cm_total_v41[i, j])
        ax.text(j, i, str(v), ha='center', va='center',
                color='white' if v > cm_total_v41.max()/2 else 'black')
plt.colorbar(im); plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'cm_v41.png'), dpi=150, bbox_inches='tight')
plt.show()

## 11. Final ranking table (sorted by accuracy)

The headline table for your report.

In [ ]:
ranking = final_df.sort_values('acc_mean', ascending=False).reset_index(drop=True)
ranking['rank']         = ranking.index + 1
ranking['acc_pct']      = (ranking['acc_mean'] * 100).round(2).astype(str) + '%'
ranking['f1_pct']       = (ranking['f1_mean']  * 100).round(2).astype(str) + '%'
ranking['params_K']     = (ranking['params']   / 1000).round(1)
ranking['flops_M']      = (ranking['flops']    / 1e6).round(3)

print_cols = ['rank', 'model', 'feature', 'params_K', 'flops_M',
              'acc_pct', 'f1_pct', 'mean_time_s']
print('FINAL RANKING (sorted by accuracy):\n')
print(ranking[print_cols].to_string(index=False))

# Save markdown version for your report
md_lines = ['| Rank | Model | Feature | Params (K) | FLOPs (M) | Accuracy | F1 |',
            '|------|-------|---------|-----------|----------|----------|------|']
for _, r in ranking.iterrows():
    md_lines.append(
        f"| {r['rank']} | {r['model']} | {r['feature']} | "
        f"{r['params_K']} | {r['flops_M']} | {r['acc_pct']} | {r['f1_pct']} |")
md_table = '\n'.join(md_lines)
with open(os.path.join(OUTPUT_DIR, 'final_ranking.md'), 'w') as f:
    f.write(md_table)
print('\nMarkdown table saved.')

# Highlight v4.1's rank
v41_rank = ranking[ranking['model'] == 'TinyPCGNet-v4.1']['rank'].values[0]
print(f'\n>>> TinyPCGNet-v4.1 ranked #{v41_rank} of {len(ranking)} <<<')

## 12. Done

Outputs in `/content/drive/MyDrive/Msc_ML_project/results_v1_final/`:

| File | Purpose |
|---|---|
| `tinypcgnet_v41_best.keras` | Input to Notebook C2 (Grad-CAM) |
| `v41_per_fold.csv` | Per-fold metrics for v4.1 |
| `v41_artifacts.npz` | Best fold's val indices + aggregate CM |
| `v41_histories.json` | Full training histories |
| `results_master_final.csv` | **22-row comparison table** |
| `final_ranking.md` | Markdown ranking table |
| `final_comparison_plots/heatmap_baselines_with_v41_in_title.png` | Heatmap |
| `final_comparison_plots/scatter_efficiency_with_v41.png` | Efficiency scatter |
| `final_comparison_plots/cm_v41.png` | v4.1 confusion matrix |

**Next:** Notebook C2 — load `tinypcgnet_v41_best.keras` and produce the
four Grad-CAM analyses (a/b/c/d) for the XAI section of your report.
